# Activation & Frequency Performance Core (Bolt Food)

Builds a scalable additive dataset and derived outputs:
- Core KPIs
- AQI (rolling 8–12w baseline)
- WoW / YoY deltas (ISO week aligned)
- Expansion-adjusted conversion
- Oaxaca–Blinder decomposition for (is_new_city × media_group)


In [ ]:
%pip install gspread gspread_dataframe

In [ ]:
import pandas as pd
import numpy as np
from datetime import date, timedelta

pd.set_option('display.max_columns', 200)


## 1) Create additive fact data from Databricks

In [ ]:
df_query = f"""
WITH tourist_info AS (
    SELECT
        uu.id AS user_id,
        MAX(CASE WHEN phone_anonymized = REPLACE(phone_prefix, '+', '') THEN 1 ELSE 0 END) AS is_local
    FROM main.ng_public.user_user uu
    LEFT JOIN main.ng_delivery.dim_delivery_city dc
        ON uu.home_city_id = dc.city_id
    LEFT JOIN main.ng_public.admin_system_country c
        ON dc.country_code = c.code
    GROUP BY 1
),

base AS (
    SELECT
        dud.user_id,
        COALESCE(dc.country_name, 'Unknown') AS country,
        COALESCE(dc.city_name, 'Unknown') AS city,
        CASE 
            WHEN dud.created_ts >= dc.food_delivery_launch_date
                AND DATEDIFF(DATE_TRUNC('WEEK', dud.created_ts),DATE_TRUNC('WEEK', dc.food_delivery_launch_date)) < 16
            THEN 1
            ELSE 0
        END AS is_new_city,


        CASE
            WHEN dud.app_install_channel_name IN ('organic', 'RiderApp') THEN dud.app_install_channel_name
            WHEN dud.app_install_channel_name IN ('Google Ads','Tiktok Ads','Facebook Ads') THEN 'social_Ads'
            --WHEN dud.app_install_channel_name IS NULL THEN 'Unknown'
            ELSE 'rest'
        END AS media_group,

        COALESCE(t.is_local, 0) AS is_local,

        dud.created_ts AS signup_ts,
        dud.delivery_activation_ts AS activation_ts,

        DATE_TRUNC('DAY', dud.created_ts) AS signup_date,
        DATE_TRUNC('DAY', dud.delivery_activation_ts) AS activation_date,

        DATEDIFF(DATE_TRUNC('DAY', dud.delivery_activation_ts), DATE_TRUNC('DAY', dud.created_ts)) AS activation_lag_days
    FROM main.ng_delivery.dim_user_delivery dud
    LEFT JOIN tourist_info t
        ON dud.user_id = t.user_id
    LEFT JOIN main.ng_delivery.dim_delivery_city dc
        ON dud.city_id = dc.city_id
),

orders_ranked AS (
    SELECT
        user_id,
        order_id,
        order_created,
        CAST(order_created AS DATE) AS order_date,
        gmv_eur,
        (bolt_delivery_campaign_cost_eur + bolt_menu_campaign_cost_eur) AS bolt_spend,
        (provider_delivery_campaign_cost_eur + provider_menu_campaign_cost_eur) AS provider_spend,
        (bolt_delivery_campaign_cost_eur + bolt_menu_campaign_cost_eur
         + provider_delivery_campaign_cost_eur + provider_menu_campaign_cost_eur) AS total_spend,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY order_created) AS order_rank
    FROM main.ng_public.etl_delivery_order_monetary_metrics
),

user_order_metrics AS (
    SELECT
        b.user_id,
        b.signup_ts,
        b.activation_ts,

        MIN(CASE WHEN o.order_rank = 1 THEN o.order_created END) AS first_order_ts,
        MAX(CASE WHEN o.order_rank = 1 THEN o.gmv_eur END) AS first_order_gmv,
        MAX(CASE WHEN o.order_rank = 1 THEN o.total_spend END) AS first_order_discount,
        MAX(CASE WHEN o.order_rank = 1 THEN o.bolt_spend END) AS first_order_discount_bolt,

        COUNT(CASE WHEN o.order_created <= DATEADD(DAY, 7, b.activation_ts) THEN o.order_id END) AS orders_7d,
        COUNT(CASE WHEN o.order_created <= DATEADD(DAY, 30, b.activation_ts) THEN o.order_id END) AS orders_30d,

        SUM(CASE WHEN o.order_created <= DATEADD(DAY, 30, b.activation_ts) THEN o.gmv_eur END) AS gmv_30d,
        SUM(CASE WHEN o.order_created <= DATEADD(DAY, 30, b.activation_ts) THEN o.total_spend END) AS discount_30d,
        SUM(CASE WHEN o.order_created <= DATEADD(DAY, 30, b.activation_ts) THEN o.bolt_spend END) AS discount_30d_bolt,

        MIN(CASE WHEN o.order_rank = 2 THEN o.order_created END) AS second_order_ts
    FROM base b
    LEFT JOIN orders_ranked o
        ON b.user_id = o.user_id
    WHERE b.activation_ts IS NOT NULL
    GROUP BY 1,2,3
),

user_metrics AS (
    SELECT
        user_id,
        CAST(TIMESTAMPDIFF(SECOND, signup_ts, first_order_ts) / 60.0 AS DOUBLE) AS time_to_first_order_minutes,
        orders_30d,
        orders_7d,
        DATEDIFF(second_order_ts, first_order_ts) AS days_to_second_order,
        CASE WHEN orders_7d >= 2 THEN 1 ELSE 0 END AS is_power_user_7d,
        CASE WHEN orders_30d >= 2 THEN 1 ELSE 0 END AS is_power_user_30d
    FROM user_order_metrics
),

signup_data AS (
    SELECT
        signup_date AS event_date,
        country, city, is_new_city, media_group, is_local,
        COUNT(DISTINCT user_id) AS signup_users
    FROM base
    GROUP BY 1,2,3,4,5,6
),

activation_data AS (
    SELECT
        b.activation_date AS event_date,
        b.country, b.city, b.is_new_city, b.media_group, b.is_local,

        COUNT(DISTINCT b.user_id) AS activated_users,
        COUNT(DISTINCT CASE WHEN b.activation_lag_days = 0 THEN b.user_id END) AS activated_users_d0,
        COUNT(DISTINCT CASE WHEN b.activation_lag_days between 1 and 14 THEN b.user_id END) AS activated_users_d14,
        COUNT(DISTINCT CASE WHEN b.activation_lag_days > 14 THEN b.user_id END) AS activated_users_d14_plus,

        SUM(uom.first_order_gmv) AS first_order_gmv,
        SUM(uom.first_order_discount) AS first_order_discount,
        SUM(uom.first_order_discount_bolt) AS first_order_discount_bolt,

        SUM(uom.gmv_30d) AS gmv_30d,
        SUM(uom.discount_30d) AS discount_30d,
        SUM(uom.discount_30d_bolt) AS discount_30d_bolt,

        SUM(um.orders_30d) AS orders_30d,
        SUM(um.days_to_second_order) AS days_to_second_order,
        SUM(um.time_to_first_order_minutes) AS time_to_first_order_minutes,
        SUM(um.time_to_first_order_minutes) FILTER (WHERE b.activation_lag_days = 0) AS time_to_first_order_minutes_d0,
        SUM(um.is_power_user_7d) AS power_users_7d,
        SUM(um.is_power_user_30d) AS power_users_30d

    FROM base b
    LEFT JOIN user_order_metrics uom
        ON b.user_id = uom.user_id
    LEFT JOIN user_metrics um
        ON b.user_id = um.user_id
    WHERE b.activation_date IS NOT NULL
    GROUP BY 1,2,3,4,5,6
)

SELECT
    COALESCE(s.event_date, a.event_date) AS event_date,
    COALESCE(s.country, a.country) AS country,
    COALESCE(s.city, a.city) AS city,
    COALESCE(s.is_new_city, a.is_new_city) AS is_new_city,
    COALESCE(s.media_group, a.media_group) AS media_group,
    COALESCE(s.is_local, a.is_local) AS is_local,

    COALESCE(s.signup_users, 0) AS signup_users,

    COALESCE(a.activated_users, 0) AS activated_users,
    COALESCE(a.activated_users_d0, 0) AS activated_users_d0,
    COALESCE(a.activated_users_d14, 0) AS activated_users_d14,
    COALESCE(a.activated_users_d14_plus, 0) AS activated_users_d14_plus,

    a.first_order_gmv,
    a.first_order_discount,
    a.first_order_discount_bolt,
    a.gmv_30d,
    a.discount_30d,
    a.discount_30d_bolt,
    a.orders_30d,
    a.days_to_second_order,
    a.time_to_first_order_minutes,
    a.time_to_first_order_minutes_d0,
    a.power_users_7d,
    a.power_users_30d
FROM signup_data s
FULL OUTER JOIN activation_data a
    ON s.event_date = a.event_date
   AND s.country = a.country
   AND s.city = a.city
   AND s.is_new_city = a.is_new_city
   AND s.media_group = a.media_group
   AND s.is_local = a.is_local
"""

df = spark.sql(df_query).toPandas()

## 2) Prep: types + week key

In [ ]:
def coerce_numeric(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def add_week(df, date_col="event_date", week_col="week"):
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])
    out[week_col] = out[date_col].dt.to_period("W").apply(lambda p: p.start_time)
    return out

SUM_COLS = [
    "signup_users","activated_users","activated_users_d0","activated_users_d14","activated_users_d14_plus",
    "first_order_gmv","first_order_discount","gmv_30d","discount_30d","orders_30d",
    "days_to_second_order","time_to_first_order_minutes","time_to_first_order_minutes_d0","power_users_7d","power_users_30d",
    "first_order_discount_bolt","discount_30d_bolt"

]

df = add_week(df)
df = coerce_numeric(df, SUM_COLS)


## 3) Aggregation + KPI helpers

In [ ]:
def safe_div(n, d):
    n = np.asarray(n, dtype="float64")
    d = np.asarray(d, dtype="float64")
    return np.where(d > 0, n / d, np.nan)

def aggregate_additive(df, dims=None, sum_cols=None, time_col="week"):
    if dims is None:
        dims = []
    if sum_cols is None:
        sum_cols = SUM_COLS
    group_cols = [time_col] + dims
    return df.groupby(group_cols, dropna=False)[sum_cols].sum().reset_index()

def add_core_kpis(df):
    out = df.copy()

    out["activation_rate"] = safe_div(out["activated_users"], out["signup_users"])
    out["same_day_activation_rate"] = safe_div(out["activated_users_d0"], out["signup_users"])
    out["d14_activation_rate"] = safe_div(out["activated_users_d14"], out["signup_users"])
    out["d14_plus_activation_rate"] = safe_div(out["activated_users_d14_plus"], out["signup_users"])

    out["power_user_rate_7d"] = safe_div(out["power_users_7d"], out["activated_users"])
    out["avg_orders_30d"] = safe_div(out["orders_30d"], out["activated_users"])

    out["first_order_discount_pct"] = safe_div(out["first_order_discount"], out["first_order_gmv"])
    out["discount_30d_pct"] = safe_div(out["discount_30d"], out["gmv_30d"])

    out["mean_time_to_first_order_minutes"] = safe_div(out["time_to_first_order_minutes"], out["activated_users"])
    out["mean_time_to_first_order_minutes_d0"] = safe_div(out["time_to_first_order_minutes_d0"], out["activated_users_d0"])
    out["mean_days_to_second_order"] = safe_div(out["days_to_second_order"], out["activated_users"])

    return out

## 4) Rolling baseline (8–12w) + AQI

In [ ]:
def rolling_window_mask(weeks: pd.Series, current_week: pd.Timestamp, window=12):
    start = current_week - pd.Timedelta(weeks=window)
    return (weeks >= start) & (weeks < current_week)

def add_aqi(
    df_week,
    group_cols=None,
    window=12,
    min_weeks=8,
    w_sameday=0.4,
    w_power=0.3,
    w_freq=0.3,
    clip=(0.5, 1.5),
):
    group_cols = list(group_cols) if group_cols else []
    #out = add_core_kpis(df_week.copy()).sort_values(group_cols + ["week"])
    out = df_week.copy().sort_values(group_cols + ["week"])
    rows = []

    # iterate per group (or once globally if no group_cols)
    groups = out.groupby(group_cols) if group_cols else [((), out)]

    for g_keys, gdf in groups:
        gdf = gdf.sort_values("week")

        for wk in gdf["week"].unique():
            hist = gdf.loc[rolling_window_mask(gdf["week"], wk, window=window)]
            if hist["week"].nunique() < min_weeks:
                continue

            base_same = np.nanmedian(hist["same_day_activation_rate"])
            base_pow  = np.nanmedian(hist["power_user_rate_7d"])
            base_freq = np.nanmedian(hist["avg_orders_30d"])

            cur = gdf.loc[gdf["week"] == wk, group_cols + [
                "week", "same_day_activation_rate", "power_user_rate_7d", "avg_orders_30d"
            ]].copy()

            cur["aqi_same_day_index"] = cur["same_day_activation_rate"] / base_same
            cur["aqi_power_index"]    = cur["power_user_rate_7d"] / base_pow
            cur["aqi_freq_index"]     = cur["avg_orders_30d"] / base_freq

            for c in ["aqi_same_day_index", "aqi_power_index", "aqi_freq_index"]:
                cur[c] = cur[c].clip(lower=clip[0], upper=clip[1])

            cur["AQI"] = (
                w_sameday * cur["aqi_same_day_index"]
                + w_power * cur["aqi_power_index"]
                + w_freq  * cur["aqi_freq_index"]
            )

            rows.append(cur[group_cols + [
                "week", "aqi_same_day_index", "aqi_power_index", "aqi_freq_index", "AQI"
            ]])

    aqi = (
        pd.concat(rows, ignore_index=True)
        if rows else
        pd.DataFrame(columns=group_cols + [
            "week", "aqi_same_day_index", "aqi_power_index", "aqi_freq_index", "AQI"
        ])
    )

    return out.merge(aqi, on=group_cols + ["week"], how="left")

## 5) WoW / YoY deltas (ISO week aligned)

In [ ]:
def add_wow_yoy(
    df,
    kpi_cols=None,
    week_col="week",
    group_cols=None,
):
    out = df.copy()
    group_cols = list(group_cols) if group_cols else []

    if kpi_cols is None:
        kpi_cols = [
            "activation_rate",
            "same_day_activation_rate",
            "d14_activation_rate",
            "d14_plus_activation_rate"
        ]

    out[week_col] = pd.to_datetime(out[week_col])
    iso = out[week_col].dt.isocalendar()
    out["iso_year"] = iso.year.astype(int)
    out["iso_week"] = iso.week.astype(int)

    out = out.sort_values(group_cols + [week_col])

    # ---- WoW ----
    for c in kpi_cols:
        if group_cols:
            out[f"{c}_wow_abs"] = out.groupby(group_cols)[c].diff(1)
            prev_val = out.groupby(group_cols)[c].shift(1)
        else:
            out[f"{c}_wow_abs"] = out[c].diff(1)
            prev_val = out[c].shift(1)

        #out[f"{c}_wow_pct"] = safe_div(out[f"{c}_wow_abs"], prev_val)

    # ---- YoY ----
    join_keys = group_cols + ["iso_year", "iso_week"]

    prev = out[join_keys + kpi_cols].copy()
    prev["iso_year"] = prev["iso_year"] + 1

    out = out.merge(
        prev,
        on=join_keys,
        how="left",
        suffixes=("", "_yoy_base")
    )

    for c in kpi_cols:
        out[f"{c}_yoy_abs"] = out[c] - out[f"{c}_yoy_base"]
        #out[f"{c}_yoy_pct"] = safe_div(out[f"{c}_yoy_abs"],out[f"{c}_yoy_base"])

    out = out.loc[:, ~out.columns.str.contains("base")]
    return out

## 6) Expansion-adjusted conversion (expected vs actual)

In [ ]:
def expansion_adjusted_expected(
    df_week_seg,
    seg_cols=("is_new_city",),
    group_cols=None,
    window=12,
    min_weeks=8,
    num_col="activated_users_d0",
    den_col="signup_users"
):
    data = df_week_seg.copy().sort_values("week")
    weeks = sorted(data["week"].unique())
    group_cols = list(group_cols) if group_cols else []

    rows = []

    for wk in weeks:
        hist = data.loc[rolling_window_mask(data["week"], wk, window=window)]
        if hist["week"].nunique() < min_weeks:
            continue

        base = (
            hist
            .groupby(group_cols + list(seg_cols))[[num_col, den_col]]
            .sum()
            .reset_index()
        )
        base["base_rate"] = safe_div(base[num_col], base[den_col])

        cur = (
            data[data["week"] == wk]
            .groupby(group_cols + list(seg_cols))[[num_col, den_col]]
            .sum()
            .reset_index()
            .merge(
                base[group_cols + list(seg_cols) + ["base_rate"]],
                on=group_cols + list(seg_cols),
                how="left"
            )
        )

        cur["expected_num"] = cur[den_col] * cur["base_rate"]

        if group_cols:
            grouped = (
                cur
                .groupby(group_cols)
                .agg(
                    signups=(den_col, "sum"),
                    expected_num=("expected_num", "sum"),
                    actual_num=(num_col, "sum")
                )
                .reset_index()
            )
        else:
            grouped = pd.DataFrame({
                "signups": [cur[den_col].sum()],
                "expected_num": [cur["expected_num"].sum()],
                "actual_num": [cur[num_col].sum()]
            })

        grouped["expected_rate"] = safe_div(grouped["expected_num"], grouped["signups"])
        grouped["actual_rate"] = safe_div(grouped["actual_num"], grouped["signups"])
        grouped["expansion_adjusted_gap_abs"] = grouped["actual_rate"] - grouped["expected_rate"]
        grouped["week"] = wk

        rows.append(grouped)

        out = pd.concat(rows, ignore_index=True)

    # ---- RENAME COLUMNS ----
    out.columns = [
        c.replace("expected_num", "expected_activations")
         .replace("actual_num", "actual_activations")
         .replace("expected_rate", "expected_same_day_activation_rate")
         .replace("actual_rate", "actual_same_day_activation_rate")
        for c in out.columns
    ]

    ordered_cols = (
        group_cols +
        ["week", "signups",
         "expected_activations", "actual_activations",
         "expected_same_day_activation_rate",
         "actual_same_day_activation_rate",
         "expansion_adjusted_gap_abs"]
    )

    return out[ordered_cols]

## 7) OB decomposition (is_new_city × media_group)

In [ ]:
def oaxaca_blinder_weekly(df_week_seg, seg_cols=("is_new_city","media_group"), window=12, min_weeks=8,
                          num_col="activated_users_d0", den_col="signup_users"):
    data = df_week_seg.copy().sort_values("week")
    weeks = sorted(data["week"].unique())

    overall_rows = []
    contrib_rows = []

    for wk in weeks:
        hist = data.loc[rolling_window_mask(data["week"], wk, window=window)]
        if len(hist["week"].unique()) < min_weeks:
            continue

        base_seg = hist.groupby(list(seg_cols))[[num_col, den_col]].sum().reset_index()
        base_seg["r_base"] = safe_div(base_seg[num_col].values, base_seg[den_col].values)

        base_tot_num = hist[num_col].sum()
        base_tot_den = hist[den_col].sum()
        r_base_total = (base_tot_num / base_tot_den) if base_tot_den > 0 else np.nan

        cur_seg = data.loc[data["week"] == wk].groupby(list(seg_cols))[[num_col, den_col]].sum().reset_index()
        cur_seg["r_cur"] = safe_div(cur_seg[num_col].values, cur_seg[den_col].values)
        cur_seg = cur_seg.merge(base_seg[list(seg_cols) + ["r_base"]], on=list(seg_cols), how="left")

        W_cur = cur_seg[den_col].sum()
        r_cur_total = (cur_seg[num_col].sum() / W_cur) if W_cur > 0 else np.nan
        mix_expected_rate = ((cur_seg[den_col] * cur_seg["r_base"]).sum() / W_cur) if W_cur > 0 else np.nan

        overall_rows.append({
            "week": wk,
            "baseline_rate": r_base_total,
            "actual_rate": r_cur_total,
            "mix_expected_rate": mix_expected_rate,
            "delta_total": r_cur_total - r_base_total,
            "mix_effect": mix_expected_rate - r_base_total,
            "behavior_effect": r_cur_total - mix_expected_rate
        })

        base_w = hist.groupby(list(seg_cols))[den_col].sum().reset_index().rename(columns={den_col: "w_base"})
        W_base = base_w["w_base"].sum()

        contrib = cur_seg.merge(base_w, on=list(seg_cols), how="left")
        contrib["w_base"] = contrib["w_base"].fillna(0)

        contrib["mix_contrib_pp"] = (
            (contrib[den_col] / W_cur) * contrib["r_base"]
            - (contrib["w_base"] / W_base) * contrib["r_base"]
        )

        for _, r in contrib.iterrows():
            row = {"week": wk, "mix_contrib_pp": r["mix_contrib_pp"]}
            for c in seg_cols:
                row[c] = r[c]
            contrib_rows.append(row)

    overall = pd.DataFrame(overall_rows)
    contrib = pd.DataFrame(contrib_rows)
    return overall, contrib

def summarize_ob_contrib(contrib_df):
    by_city = contrib_df.groupby(["week","is_new_city"])["mix_contrib_pp"].sum().reset_index()
    by_media = contrib_df.groupby(["week","media_group"])["mix_contrib_pp"].sum().reset_index()
    return by_city, by_media

## 8.1) Run the pipeline (weekly global outputs)

Edit `FILTER_COUNTRIES` and `DATE_CUTOFF` then run. 

The recommended export targets for Sheets are: `weekly_global`, `exp_adj`, `ob_overall`, `ob_by_city`, `ob_by_media`.

In [ ]:
df = add_week(df)
df = coerce_numeric(df, SUM_COLS)

In [ ]:
FILTER_COUNTRIES = [
    "Estonia", "Latvia", "Lithuania", "Slovakia", "Czech Republic", "Malta",
    "Romania", "Poland", "Portugal", "Ukraine", "Cyprus",
    "Bulgaria", "Azerbaijan", "Ghana", "Kenya"
]
DATE_CUTOFF = "2024-01-01"
DIMENSIONS = ["is_new_city","media_group"]

filtered = df[(df["event_date"] >= DATE_CUTOFF) & (df["country"].isin(FILTER_COUNTRIES))].copy()

# 1) weekly global (no segments)
weekly_global = aggregate_additive(filtered, dims=[], sum_cols=SUM_COLS, time_col="week")
weekly_global = add_core_kpis(weekly_global.copy()).sort_values(["week"])
weekly_global = add_aqi(weekly_global, window=12, min_weeks=8)
weekly_global = add_wow_yoy(weekly_global, group_cols=None)

# 2) weekly segmented for expansion + OB
weekly_seg = aggregate_additive(filtered, dims=["is_new_city","media_group"], sum_cols=SUM_COLS, time_col="week")

exp_adj = expansion_adjusted_expected(weekly_seg, seg_cols=("is_new_city",), window=12, min_weeks=8)
ob_overall, ob_contrib = oaxaca_blinder_weekly(weekly_seg, seg_cols=("is_new_city","media_group"), window=12, min_weeks=8)
ob_by_city, ob_by_media = summarize_ob_contrib(ob_contrib)

## 8.2) Run the pipeline (weekly country outputs)

Edit `FILTER_COUNTRIES` and `DATE_CUTOFF` then run. 

The recommended export targets for Sheets are: `weekly_global`, `exp_adj`, `ob_overall`, `ob_by_city`, `ob_by_media`.

In [ ]:
# 1) weekly Country (no segments)
DIMENSIONS = ["country"]
weekly_country = aggregate_additive(filtered, dims=["country"], sum_cols=SUM_COLS, time_col="week")
weekly_country = add_core_kpis(weekly_country.copy()).sort_values(["country","week"])
weekly_country = add_aqi(weekly_country, window=12, min_weeks=8, group_cols=("country",))
weekly_country = add_wow_yoy(weekly_country, group_cols=("country",))

# 2) weekly segmented for expansion
weekly_country_seg = aggregate_additive(filtered, dims=["country","is_new_city","media_group"], sum_cols=SUM_COLS, time_col="week")

weekly_country_exp_adj = expansion_adjusted_expected(weekly_country_seg,seg_cols=("is_new_city",),group_cols=("country",))

## 8.3) Run the pipeline (weekly city outputs)

Edit `FILTER_COUNTRIES` and `DATE_CUTOFF` then run. 

The recommended export targets for Sheets are: `weekly_global`, `exp_adj`, `ob_overall`, `ob_by_city`, `ob_by_media`.

In [ ]:
active_city_query = f"""
select city_name as city
from ng_delivery_spark.dim_delivery_city
where food_delivery_launch_date is not null
group by 1
"""
active_cities_df = spark.sql(active_city_query).toPandas()

In [ ]:
# 1) weekly city (with segs)

city_dims = ["country","city","is_new_city","is_local","media_group"]
weekly_city = aggregate_additive(filtered, dims=city_dims, sum_cols=SUM_COLS, time_col="week")
weekly_city = add_core_kpis(weekly_city.copy()).sort_values(["country","city","is_new_city","is_local","media_group","week"])
weekly_city = add_wow_yoy(weekly_city, group_cols=(["country","city","is_new_city","is_local","media_group"]))

# 2) Remove cities that are now launched & filter to last 12 weeks

FILTER_CITIES = active_cities_df["city"].unique()
today = pd.Timestamp.today().normalize()

weekly_city = weekly_city[
    (weekly_city["city"].isin(FILTER_CITIES)) &
    (
        (weekly_city["week"] >= today - pd.Timedelta(weeks=12)) |
        (
            (weekly_city["week"] >= today - pd.Timedelta(weeks=64)) &
            (weekly_city["week"] <  today - pd.Timedelta(weeks=52))
        )
    )
]

## 10) Global Counterfactuals

Compute `mix` and `behaviour` impacts to highlight cuase for change in first day conversion.

* Country Mix Impact
* New vs Old City Mix Impact
* Local vs Tourist
* Marketing Mix

In [ ]:
df_query_country_mix_impact = f"""
WITH base AS (
    SELECT
        dud.user_id,
        dc.country_name AS country,
        DATE_TRUNC('WEEK', dud.created_ts) AS week,
        CASE
            WHEN DATE_TRUNC('DAY', dud.delivery_activation_ts)
               = DATE_TRUNC('DAY', dud.created_ts)
            THEN 1 ELSE 0
        END AS is_same_day_activated
    FROM ng_delivery_spark.dim_user_delivery dud
    JOIN ng_delivery_spark.dim_delivery_city dc
        ON dud.city_id = dc.city_id
    WHERE dud.created_ts IS NOT NULL
        AND dc.country_name IN (
       'Estonia', 'Latvia', 'Lithuania', 'Slovakia', 'Czech Republic', 'Malta',
        'Romania', 'Poland', 'Portugal', 'Ukraine', 'Cyprus',
        'Bulgaria', 'Azerbaijan', 'Ghana', 'Kenya')
),

weekly_country AS (
    SELECT
        week,
        country,
        COUNT(DISTINCT user_id) AS signup_users,
        SUM(is_same_day_activated) AS same_day_activations,
        SUM(is_same_day_activated) * 1.0 / NULLIF(signup_users, 0) AS same_day_activation_rate
    FROM base
    GROUP BY 1,2
),

weekly_with_shares AS (
    SELECT
        *,
        signup_users * 1.0
            / SUM(signup_users) OVER (PARTITION BY week)
            AS signup_share
    FROM weekly_country
),

weekly_yoy AS (
    SELECT
        cur.week,
        cur.country,

        cur.signup_users,
        cur.signup_share,
        cur.same_day_activation_rate,

        prev.signup_users AS signup_users_yoy,
        prev.signup_share AS signup_share_yoy,
        prev.same_day_activation_rate AS same_day_activation_rate_yoy

    FROM weekly_with_shares cur
    LEFT JOIN weekly_with_shares prev
        ON cur.country = prev.country
       AND cur.week = DATEADD(WEEK, 52, prev.week)
)

SELECT *
FROM weekly_yoy
WHERE week >= DATE '2024-01-01'
ORDER BY week, country;


"""

df_country_mix_impact = spark.sql(df_query_country_mix_impact).toPandas()

In [ ]:
df_query_city_mix_impact = f"""
WITH base AS (
    SELECT
        dud.user_id,

        CASE 
            WHEN dud.created_ts >= dc.food_delivery_launch_date
             AND DATEDIFF(
                 DATE_TRUNC('WEEK', dud.created_ts),
                 DATE_TRUNC('WEEK', dc.food_delivery_launch_date)
             ) < 16
            THEN TRUE
            ELSE FALSE
        END AS is_new_city,

        DATE_TRUNC('WEEK', dud.created_ts) AS week,

        CASE
            WHEN DATE_TRUNC('DAY', dud.delivery_activation_ts)
               = DATE_TRUNC('DAY', dud.created_ts)
            THEN 1 ELSE 0
        END AS is_same_day_activated

    FROM ng_delivery_spark.dim_user_delivery dud
    JOIN ng_delivery_spark.dim_delivery_city dc
        ON dud.city_id = dc.city_id
    WHERE dud.created_ts IS NOT NULL 
        and dc.country_name IN (
                'Estonia', 'Latvia', 'Lithuania', 'Slovakia', 'Czech Republic', 'Malta',
                'Romania', 'Poland', 'Portugal', 'Ukraine', 'Cyprus',
                'Bulgaria', 'Azerbaijan', 'Ghana', 'Kenya')
),

weekly_city_type AS (
    SELECT
        week,
        is_new_city,
        COUNT(DISTINCT user_id) AS signup_users,
        SUM(is_same_day_activated) AS same_day_activations,
        SUM(is_same_day_activated) * 1.0
            / NULLIF(COUNT(DISTINCT user_id), 0) AS same_day_activation_rate
    FROM base
    GROUP BY 1,2
),

weekly_with_shares AS (
    SELECT
        *,
        signup_users * 1.0
            / SUM(signup_users) OVER (PARTITION BY week)
            AS signup_share
    FROM weekly_city_type
),

weekly_yoy AS (
    SELECT
        cur.week,
        cur.is_new_city,

        cur.signup_users,
        cur.signup_share,
        cur.same_day_activation_rate,

        prev.signup_users AS signup_users_yoy,
        prev.signup_share AS signup_share_yoy,
        prev.same_day_activation_rate AS same_day_activation_rate_yoy

    FROM weekly_with_shares cur
    LEFT JOIN weekly_with_shares prev
        ON cur.is_new_city = prev.is_new_city
       AND cur.week = DATEADD(WEEK, 52, prev.week)
)

SELECT *
FROM weekly_yoy
WHERE week >= DATE '2024-01-01'
ORDER BY week, is_new_city;

"""

df_city_mix_impact = spark.sql(df_query_city_mix_impact).toPandas()


In [ ]:
df_query_tourist_mix_impact = f"""
WITH tourist_info AS (
    SELECT
        uu.id AS user_id,
        MAX(
            CASE
                WHEN phone_anonymized = REPLACE(phone_prefix, '+', '')
                THEN TRUE ELSE FALSE
            END
        ) AS is_local
    FROM ng_public_spark.user_user uu
    LEFT JOIN ng_delivery_spark.dim_delivery_city dc
        ON uu.home_city_id = dc.city_id
    LEFT JOIN ng_public_spark.admin_system_country c
        ON dc.country_code = c.code
    GROUP BY 1
),

base AS (
    SELECT
        dud.user_id,
        COALESCE(t.is_local, FALSE) AS is_local,
        DATE_TRUNC('WEEK', dud.created_ts) AS week,

        CASE
            WHEN DATE_TRUNC('DAY', dud.delivery_activation_ts)
               = DATE_TRUNC('DAY', dud.created_ts)
            THEN 1 ELSE 0
        END AS is_same_day_activated

    FROM ng_delivery_spark.dim_user_delivery dud
    JOIN ng_delivery_spark.dim_delivery_city dc
        ON dud.city_id = dc.city_id
    LEFT JOIN tourist_info t
        ON dud.user_id = t.user_id
    WHERE dud.created_ts IS NOT NULL
        and dc.country_name IN (
                'Estonia', 'Latvia', 'Lithuania', 'Slovakia', 'Czech Republic', 'Malta',
                'Romania', 'Poland', 'Portugal', 'Ukraine', 'Cyprus',
                'Bulgaria', 'Azerbaijan', 'Ghana', 'Kenya')
),

weekly_locality AS (
    SELECT
        week,
        is_local,
        COUNT(DISTINCT user_id) AS signup_users,
        SUM(is_same_day_activated) AS same_day_activations,
        SUM(is_same_day_activated) * 1.0
            / NULLIF(COUNT(DISTINCT user_id), 0) AS same_day_activation_rate
    FROM base
    GROUP BY 1,2
),

weekly_with_shares AS (
    SELECT
        *,
        signup_users * 1.0
            / SUM(signup_users) OVER (PARTITION BY week)
            AS signup_share
    FROM weekly_locality
),

weekly_yoy AS (
    SELECT
        cur.week,
        cur.is_local,

        cur.signup_users,
        cur.signup_share,
        cur.same_day_activation_rate,

        prev.signup_users AS signup_users_yoy,
        prev.signup_share AS signup_share_yoy,
        prev.same_day_activation_rate AS same_day_activation_rate_yoy

    FROM weekly_with_shares cur
    LEFT JOIN weekly_with_shares prev
        ON cur.is_local = prev.is_local
       AND cur.week = DATEADD(WEEK, 52, prev.week)
)

SELECT *
FROM weekly_yoy
WHERE week >= DATE '2024-01-01'
ORDER BY week, is_local;
"""

df_tourist_mix_impact = spark.sql(df_query_tourist_mix_impact).toPandas()


In [ ]:
df_query_marketing_mix_impact = f"""
WITH base AS (
    SELECT
        dud.user_id,

        CASE
            WHEN dud.app_install_channel_name IN ('organic', 'RiderApp')THEN dud.app_install_channel_name
            WHEN dud.app_install_channel_name IN ('Google Ads','Tiktok Ads','Facebook Ads') THEN 'social_Ads'
            --WHEN dud.app_install_channel_name IS NULL THEN 'Unknown'
            ELSE 'rest'
        END AS media_group,

        DATE_TRUNC('WEEK', dud.created_ts) AS week,

        CASE
            WHEN DATE_TRUNC('DAY', dud.delivery_activation_ts)
               = DATE_TRUNC('DAY', dud.created_ts)
            THEN 1 ELSE 0
        END AS is_same_day_activated

    FROM ng_delivery_spark.dim_user_delivery dud
    JOIN ng_delivery_spark.dim_delivery_city dc
        ON dud.city_id = dc.city_id
    WHERE dud.created_ts IS NOT NULL
        and dc.country_name IN (
                'Estonia', 'Latvia', 'Lithuania', 'Slovakia', 'Czech Republic', 'Malta',
                'Romania', 'Poland', 'Portugal', 'Ukraine', 'Cyprus',
                'Bulgaria', 'Azerbaijan', 'Ghana', 'Kenya')
),

weekly_media AS (
    SELECT
        week,
        media_group,
        COUNT(DISTINCT user_id) AS signup_users,
        SUM(is_same_day_activated) AS same_day_activations,
        SUM(is_same_day_activated) * 1.0
            / NULLIF(COUNT(DISTINCT user_id), 0) AS same_day_activation_rate
    FROM base
    GROUP BY 1,2
),

weekly_with_shares AS (
    SELECT
        *,
        signup_users * 1.0
            / SUM(signup_users) OVER (PARTITION BY week)
            AS signup_share
    FROM weekly_media
),

weekly_yoy AS (
    SELECT
        cur.week,
        cur.media_group,

        cur.signup_users,
        cur.signup_share,
        cur.same_day_activation_rate,

        prev.signup_users AS signup_users_yoy,
        prev.signup_share AS signup_share_yoy,
        prev.same_day_activation_rate AS same_day_activation_rate_yoy

    FROM weekly_with_shares cur
    LEFT JOIN weekly_with_shares prev
        ON cur.media_group = prev.media_group
       AND cur.week = DATEADD(WEEK, 52, prev.week)
)

SELECT *
FROM weekly_yoy
WHERE week >= DATE '2024-01-01'
ORDER BY week, media_group;

"""

df_marketing_mix_impact = spark.sql(df_query_marketing_mix_impact).toPandas()


## 9) Export to Google Sheets (optional)

In [ ]:
import json 
import gspread as gs
from gspread_dataframe import set_with_dataframe

def export_df_to_gsheet(df, spreadsheet_key, worksheet_name):
    # Ensure datetime columns are date-only
    df = df.copy()
    for col in df.select_dtypes(include=["datetime64[ns]"]).columns:
        df[col] = df[col].dt.date

    credentials = json.loads(dbutils.secrets.get(scope="google_sheet", key="user"))
    gc = gs.service_account_from_dict(credentials)
    sh = gc.open_by_key(spreadsheet_key)

    try:
        ws = sh.worksheet(worksheet_name)
        ws.clear()
    except Exception:
        ws = sh.add_worksheet(
            title=worksheet_name,
            rows=str(len(df) + 10),
            cols=str(len(df.columns) + 10)
        )

    set_with_dataframe(ws, df, include_index=False, resize=True)

#
spreadsheet_key = '1nOB9mk_mRJesDuvnyAYGZoCEaasgQLG5CI_l-WjsV90' 

# EXPORT GLOBAL INFO
export_df_to_gsheet(weekly_global, spreadsheet_key, worksheet_name="weekly_global")
export_df_to_gsheet(exp_adj, spreadsheet_key, worksheet_name="exp_adj_global")
export_df_to_gsheet(ob_overall, spreadsheet_key, worksheet_name="ob_overall")
export_df_to_gsheet(ob_by_city, spreadsheet_key, worksheet_name="ob_by_city")
export_df_to_gsheet(ob_by_media, spreadsheet_key, worksheet_name="ob_by_media")

# EXPORT COUNTRY INFO
export_df_to_gsheet(weekly_country, spreadsheet_key, worksheet_name="weekly_country")
export_df_to_gsheet(weekly_country_exp_adj, spreadsheet_key, worksheet_name="exp_adj_country")

# EXPORT CITY INFO
export_df_to_gsheet(weekly_city, spreadsheet_key, worksheet_name="weekly_city")

# EXPORT COUNTERFACTUALS OF CONVERSION 
export_df_to_gsheet(df_country_mix_impact, spreadsheet_key, worksheet_name="country_mix_impact")
export_df_to_gsheet(df_city_mix_impact, spreadsheet_key, worksheet_name="city_mix_impact")
export_df_to_gsheet(df_tourist_mix_impact, spreadsheet_key, worksheet_name="tourist_mix_impact")
export_df_to_gsheet(df_marketing_mix_impact, spreadsheet_key, worksheet_name="marketing_mix_impact")

Miscalleneous